In [ ]:
import os
print(os.getcwd())

In [ ]:
import os
print(os.listdir('/content'))


In [ ]:
import pandas as pd

clientes = pd.read_csv("/content/clientes.csv")
produtos = pd.read_csv("/content/produtos.csv")
calendario = pd.read_csv("/content/dim_calendario.csv")
vendas = pd.read_csv("/content/vendas_zoop.csv")
pagamentos = pd.read_csv("/content/pagamentos_api.csv")
reclamacoes = pd.read_csv("/content/reclamacoes_clientes.csv")

In [ ]:
print(clientes.shape)
print(produtos.shape)
print(vendas.shape)
print(pagamentos.shape)
print(reclamacoes.shape)

In [ ]:
print(vendas.info())
print(vendas.isnull().sum())

print(vendas["id_venda"].duplicated().sum())
print(pagamentos["transaction_id"].duplicated().sum())


In [ ]:
print(vendas["id_cliente"].isin(clientes["id_cliente"]).mean())
print(vendas["id_produto"].isin(produtos["id_produto"]).mean())
print(pagamentos["id_venda"].isin(vendas["id_venda"]).mean())

In [ ]:
vendas["receita_bruta"] = vendas["quantidade"] * vendas["preco_unitario"]

vendas["valor_desconto"] = vendas["receita_bruta"] * (vendas["desconto_pct"] / 100)

vendas["valor_previsto_calc"] = (
    vendas["receita_bruta"]
    - vendas["valor_desconto"]
    + vendas["valor_frete"]
)

In [ ]:
df = vendas.merge(pagamentos, on="id_venda", how="left")

df["delta"] = (df["valor_previsto"] - df["valor_previsto_calc"]).abs()

df["flag_divergencia"] = df["delta"] > 0.01

In [ ]:
df = df.merge(clientes, on="id_cliente", how="left")
df = df.merge(produtos, on="id_produto", how="left")
df = df.merge(calendario, left_on="data_venda", right_on="data", how="left")

In [ ]:
df["margem_bruta"] = df["receita_bruta"] - (df["custo"] * df["quantidade"])

In [ ]:
df["R01"] = ((df["hora"] >= 0) & (df["hora"] <= 5) & (df["valor_previsto_calc"] > 1500)).astype(int)

df["R02"] = ((df["valor_pago"] - df["valor_previsto"]).abs() /
             df["valor_previsto"].clip(lower=0.01)) > 0.10

df["R03"] = (df["status"] == "chargeback").astype(int)

df["R04"] = df["transaction_id"].duplicated(keep=False).astype(int)

df["score_risco"] = 3*df["R01"] + 4*df["R02"] + 5*df["R03"] + 4*df["R04"]

In [ ]:
print(df.columns.tolist())

In [ ]:
df[["hora_venda", "horario_transacao"]].head()

In [ ]:
import pandas as pd

df["hora_venda"] = pd.to_datetime(df["hora_venda"], format="%H:%M:%S")

df["hora"] = df["hora_venda"].dt.hour

In [ ]:
df["R01"] = (
    (df["hora"] >= 0) &
    (df["hora"] <= 5) &
    (df["valor_previsto_calc"] > 1500)
).astype(int)

In [ ]:
df["R02"] = (
    ((df["valor_pago"] - df["valor_previsto"]).abs() /
     df["valor_previsto"].clip(lower=0.01)) > 0.10
).astype(int)

df["R03"] = (df["status"] == "chargeback").astype(int)

df["R04"] = df["transaction_id"].duplicated(keep=False).astype(int)

df["score_risco"] = (
    3 * df["R01"] +
    4 * df["R02"] +
    5 * df["R03"] +
    4 * df["R04"]
)

In [ ]:
df[["R01", "R02", "R03", "R04", "score_risco"]].head()

In [ ]:
top200 = df.sort_values("score_risco", ascending=False).head(200)

top200.to_csv("top200_suspeitas.csv", index=False)

In [ ]:
df.to_csv("fato_vendas_pagamentos.csv", index=False)

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
from google.colab import files

files.download("fato_vendas_pagamentos.csv")
files.download("top200_suspeitas.csv")